# Evaluación de los brazos A / B

Notebook de evaluación: carga un `best.pt` previamente entrenado y reporta
mAP50, mAP50-95, precisión y recall por clase, junto con las curvas PR y las
cajas predichas sobre el split de validación.

## Uso

1. Ajustar `WEIGHTS` y `TEST_YAML` en la celda de configuración.
2. Ejecutar las celdas en orden.

Para el flujo completo (descarga → etiquetado → entrenamiento) utilizar
`training/pipeline_completo.ipynb`.


In [ ]:
# --- Configuración ---------------------------------------------------------
import json
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vchacin/AnyoneAI-object-detection.git"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics", "pandas", "pillow", "matplotlib",
                    "opencv-python-headless", "pyyaml", "tqdm"], check=True)
    ROOT_DIR = Path("/content/repo")
    if not ROOT_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch",
                        REPO_URL, str(ROOT_DIR)], check=True)
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "training").is_dir() and (ROOT_DIR.parent / "training").is_dir():
        ROOT_DIR = ROOT_DIR.parent

CONFIGS_DIR = ROOT_DIR / "training" / "configs"
WEIGHTS_DIR = ROOT_DIR / "training" / "weights"
REPORTS_DIR = ROOT_DIR / "training" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = WEIGHTS_DIR / "A_e35" / "weights" / "best.pt"
TEST_YAML = CONFIGS_DIR / "test_obj.yaml"      # test_2cls.yaml para el brazo B
IMGSZ = 1024
MAX_DET = 750

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

assert WEIGHTS.is_file(), f"No se encuentra {WEIGHTS}. Entrenar primero con pipeline_completo.ipynb."
print("Pesos:", WEIGHTS)


In [ ]:
# --- Métricas --------------------------------------------------------------
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
metrics = model.val(data=str(TEST_YAML), imgsz=IMGSZ, batch=1,
                    max_det=MAX_DET, project=str(REPORTS_DIR),
                    name=WEIGHTS.parent.parent.name + "_val", exist_ok=True)

box = metrics.box
summary = {
    "weights": str(WEIGHTS),
    "data": str(TEST_YAML),
    "results": {k: round(float(v), 5) for k, v in metrics.results_dict.items()},
    "per_class": {
        "names": list(box.names.values()) if hasattr(box, "names") else None,
        "precision": [round(float(v), 5) for v in box.p],
        "recall": [round(float(v), 5) for v in box.r],
        "ap50": [round(float(v), 5) for v in box.ap50],
        "map50_95": [round(float(v), 5) for v in box.maps],
    },
}
print(json.dumps(summary, indent=2))

(REPORTS_DIR / "evaluation.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8")


In [ ]:
# --- Curvas y predicciones sobre el split de validación ---------------------
import cv2
import numpy as np
from api.gap_heuristic import score_candidates

val_dir = Path(metrics.save_dir)
print("Artefactos de val en:", val_dir)
for name in ("PR_curve.png", "F1_curve.png", "confusion_matrix.png"):
    p = val_dir / name
    print(("✅" if p.is_file() else "❌"), name)

# Cajas predichas + huecos heurísticos sobre las primeras imágenes del val
IMAGES_DIR = ROOT_DIR / "data_pipeline" / "dataset_sku110k" / "images"
for path in sorted(IMAGES_DIR.glob("*.jpg"))[:5]:
    result = model.predict(str(path), conf=0.25, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy().tolist()
    image = cv2.imread(str(path))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    candidates = score_candidates(gray, boxes, auto_score=0.5)
    print(f"{path.name}: {len(boxes)} productos, "
          f"{len(candidates)} huecos candidatos")
